In [1]:
import os
import pickle
import base64
import time
import json
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [2]:
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']
DOWNLOAD_DIR = 'parent_directory'
CHECKPOINT_FILE = 'checkpoint.json'

In [3]:
def retry_with_backoff(func, max_retries=5):
    """Retry a function with exponential backoff on connection or 429 errors."""
    for attempt in range(max_retries):
        try:
            return func()
        except (ConnectionError, HttpError) as e:
            if isinstance(e, HttpError) and e.resp.status != 429:
                raise  # re-raise if not a quota error
            wait = (2 ** attempt) + 1  # 2,4,8,16,32 seconds
            print(f"Retry {attempt+1}/{max_retries} after {wait}s due to: {e}")
            time.sleep(wait)
    raise Exception("Max retries exceeded")

In [4]:
def get_gmail_service():
    creds = None
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)
    return build('gmail', 'v1', credentials=creds)

In [5]:
def list_messages_page(service, page_token=None):
    """Fetch one page of messages with retry logic."""
    def _list():
        return service.users().messages().list(
            userId='me', pageToken=page_token, maxResults=500
        ).execute()
    return retry_with_backoff(_list)

In [6]:
def get_message(service, msg_id):
    def _get():
        return service.users().messages().get(userId='me', id=msg_id).execute()
    return retry_with_backoff(_get)

In [7]:
def get_attachment(service, msg_id, att_id):
    def _get():
        return service.users().messages().attachments().get(
            userId='me', messageId=msg_id, id=att_id
        ).execute()
    return retry_with_backoff(_get)

In [8]:
def process_parts(service, msg_id, parts, download_dir):
    """Recursively process message parts to find attachments."""
    for part in parts:
        if 'parts' in part:
            process_parts(service, msg_id, part['parts'], download_dir)
        else:
            if 'filename' in part and part['filename'] and 'attachmentId' in part.get('body', {}):
                filename = part['filename']
                att_id = part['body']['attachmentId']
                # Skip if file already exists (optional)
                safe_name = f"{msg_id}_{filename}"
                filepath = os.path.join(download_dir, safe_name)
                if os.path.exists(filepath):
                    print(f"Skipping {safe_name} (already exists)")
                    continue
                # Download attachment
                attachment = get_attachment(service, msg_id, att_id)
                data = attachment.get('data')
                if data:
                    file_data = base64.urlsafe_b64decode(data.encode('utf-8'))
                    with open(filepath, 'wb') as f:
                        f.write(file_data)
                    print(f"Downloaded: {safe_name}")

In [9]:
def main():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    service = get_gmail_service()

    # Load checkpoint if exists
    checkpoint = {}
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            checkpoint = json.load(f)
    next_page = checkpoint.get('next_page_token', None)
    processed_count = checkpoint.get('processed_count', 0)

    print(f"Resuming from processed_count={processed_count}, page_token={next_page}")

    while True:
        # Fetch a page of messages
        results = list_messages_page(service, next_page)
        messages = results.get('messages', [])
        if not messages:
            break

        for msg in messages:
            processed_count += 1
            msg_id = msg['id']
            print(f"Processing message {processed_count}: {msg_id}")
            # Retrieve full message
            message = get_message(service, msg_id)
            payload = message.get('payload', {})
            # Process parts (handles nested MIME)
            if 'parts' in payload:
                process_parts(service, msg_id, payload['parts'], DOWNLOAD_DIR)
            elif 'attachmentId' in payload.get('body', {}):
                # Single part attachment
                process_parts(service, msg_id, [payload], DOWNLOAD_DIR)
            # Small delay to avoid rate limits
            time.sleep(0.5)

        # Save checkpoint after each page
        next_page = results.get('nextPageToken')
        checkpoint = {
            'next_page_token': next_page,
            'processed_count': processed_count
        }
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump(checkpoint, f)

        if not next_page:
            break

    print("All messages processed. Checkpoint file remains; delete it when done.")

In [10]:
if __name__ == '__main__':
    main()

Resuming from processed_count=500, page_token=17027044715268293202
Processing message 501: 19e87ab5f22f2438
Skipping 19e87ab5f22f2438_1_5_ZAYAD_9_UTTAR PRADESH_66_Chandauli_1_1_2025-26_Chakia_Variyarpur.zip (already exists)
Skipping 19e87ab5f22f2438_1_5_ZAYAD_9_UTTAR PRADESH_66_Chandauli_1_2_2025-26_Chakia_Akodhwa Mu. Kodochak.zip (already exists)
Processing message 502: 19e87aa641c46ca2
Skipping 19e87aa641c46ca2_1_5_ZAYAD_9_UTTAR PRADESH_66_Chandauli_2_2_2025-26_Chandauli_Matkkipur.zip (already exists)
Skipping 19e87aa641c46ca2_1_5_ZAYAD_9_UTTAR PRADESH_65_Ghazipur_4_1_2025-26_Saidpur_Bhagwanpur.zip (already exists)
Skipping 19e87aa641c46ca2_1_5_ZAYAD_9_UTTAR PRADESH_65_Ghazipur_4_2_2025-26_Saidpur_Jaitwar Savana.zip (already exists)
Skipping 19e87aa641c46ca2_1_5_ZAYAD_9_UTTAR PRADESH_65_Ghazipur_4_3_2025-26_Saidpur_Salone Patti.zip (already exists)
Processing message 503: 19e879c1627ac804
Skipping 19e879c1627ac804_1_5_SUMMER_24_GUJARAT_1_KACHCHH_2_1_2025-26_NAKHATRANA_Bibar.zip (alre